# CropDetector — Phase 9, Part 4
## Independent stress testing of the Stage A plant / non-plant validator

**Standalone Google Colab notebook.** Upload only your saved `CropDetector_Phase9_Part3_Results.zip`. No previous cells, retraining or original dataset re-downloads are needed.

**Why this step matters:** The Part 3 developer split gave 100% accuracy, but all plants were from PlantDoc and all non-plants were from Natural Images. We need to check behaviour on photographs from independent sources **before** INT8 quantization or Android integration.

**Independent diagnostic datasets:** [Beans (field leaf photographs)](https://www.tensorflow.org/datasets/catalog/beans), [TF Flowers](https://www.tensorflow.org/datasets/catalog/tf_flowers), [CIFAR-10 test split](https://www.tensorflow.org/api_docs/python/tf/keras/datasets/cifar10/load_data). CIFAR images are only 32×32, so they are **low-resolution stress tests, not representative mobile-camera benchmarks**. Dataset provenance, image content and the assumptions about positive/negative labels limit interpretation.

We will **not tune or change thresholds based on the external stress sets**. We will compare two frozen settings: the ordinary `plant_score >= 0.5` decision rule and the saved Part 3 high-retention threshold.

## Cell 1 — Environment and reproducibility

In [ ]:
%pip -q install tensorflow-datasets

import io, os, json, zipfile, hashlib, shutil, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from scipy.special import expit
from sklearn.metrics import roc_auc_score, confusion_matrix
from IPython.display import display
import tensorflow as tf
import tensorflow_datasets as tfds
from google.colab import files

SEED = 42
IMG_SIZE = (160,160)
BATCH = 32
ROOT = Path('/content')
OUT = ROOT/'CropDetector_Phase9_Part4'
OUT.mkdir(exist_ok=True, parents=True)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow',tf.__version__,'TFDS',tfds.__version__)
print('Accelerators',tf.config.list_physical_devices('GPU'))

## Cell 2 — Upload the Part 3 results ZIP and verify the original model

Upload **`CropDetector_Phase9_Part3_Results.zip`**. The model SHA-256 must match the saved configuration.

In [ ]:
PART3 = ROOT/'CropDetector_Phase9_Part3_Results.zip'
if not PART3.exists():
    uploaded=files.upload()
    assert PART3.name in uploaded, 'Please upload CropDetector_Phase9_Part3_Results.zip'

with zipfile.ZipFile(PART3) as z:
    needed={'crop_validator_stageA.keras','stageA_config.json','training_summary.json','development_logits.npy'}
    assert needed.issubset(z.namelist()), f'Missing: {needed-set(z.namelist())}'
    config=json.loads(z.read('stageA_config.json'))
    dev_summary=json.loads(z.read('training_summary.json'))
    model_data=z.read('crop_validator_stageA.keras')
    (OUT/'crop_validator_stageA.keras').write_bytes(model_data)
    frozen_dev_logits=np.load(io.BytesIO(z.read('development_logits.npy')),allow_pickle=False)

model_sha=hashlib.sha256(model_data).hexdigest()
assert model_sha == config['model_sha256']
assert len(frozen_dev_logits)==1012
assert config['input_shape']==[1,160,160,3]
assert config['output']=='one logit for Non-plant'

model=tf.keras.models.load_model(str(OUT/'crop_validator_stageA.keras'),compile=False)
assert tuple(model.input_shape[1:])==(160,160,3)
assert tuple(model.output_shape[1:])==(1,)
model.trainable=False

FROZEN_THRESHOLDS={
    'Default decision (0.5)': 0.5,
    'Part 3 operating threshold': float(config['accept_if_plant_score_at_least'])
}
print('Model SHA256:',model_sha)
print('Logit convention: positive = non-plant')
print('Frozen thresholds:',FROZEN_THRESHOLDS)
print('Part 3 selected checkpoint:',config['selected_checkpoint'])
print('Part 3 fine-tuned development metrics:')
display(pd.DataFrame(dev_summary['model_comparison']).round(4))
print('Part 3 results at operating threshold:')
display(pd.DataFrame(dev_summary['per_role']))

## Cell 3 — Independent **positive** examples: field bean leaves

The official Beans dataset contains smartphone photographs of bean leaves affected by disease or healthy. We use the **test split** only. It was not used in Part 3 training.

In [ ]:
samples=[]
images=[]

def add_sample(arr, source, category, truth, source_index):
    # Every image gets the same deterministic RGB/BILINEAR preprocessing.
    im=Image.fromarray(np.asarray(arr,dtype=np.uint8)).convert('RGB')
    im=im.resize((160,160),resample=Image.Resampling.BILINEAR)
    rgb=np.asarray(im,dtype=np.uint8)
    digest=hashlib.sha256(rgb.tobytes()).hexdigest()
    images.append(rgb)
    samples.append({'Source':source,'Category':str(category),
                    'Truth':int(truth),'SourceIndex':int(source_index),
                    'ImageSHA256_RGB160':digest})

beans_ds,beans_info=tfds.load('beans',split='test',as_supervised=True,
                              with_info=True,shuffle_files=False)
for i,(img,label) in enumerate(tfds.as_numpy(beans_ds)):
    class_name=beans_info.features['label'].int2str(int(label))
    add_sample(img,'Beans / independent field leaves',class_name,0,i)

print('Bean leaf examples:',len(samples))

## Cell 4 — Independent **positive** examples: flowers

The TF Flowers dataset contains 3,670 images. Select 300 indices with a fixed seed from its full train split. Here *train* is the **source dataset's only public split**, not a training split for our validator: these images are used solely for stress evaluation.

In [ ]:
flower_rng=np.random.default_rng(SEED)
flower_ds,flower_info=tfds.load('tf_flowers',split='train',as_supervised=True,
                               with_info=True,shuffle_files=False)
flower_n=flower_info.splits['train'].num_examples
chosen=set(map(int,flower_rng.choice(flower_n,size=min(300,flower_n),replace=False)))
count_before=len(samples)
for i,(img,label) in enumerate(tfds.as_numpy(flower_ds)):
    if i in chosen:
        category=flower_info.features['label'].int2str(int(label))
        add_sample(img,'TF Flowers / external flowers',category,0,i)
print('Flower examples:',len(samples)-count_before)
assert len(samples)-count_before==len(chosen)

## Cell 5 — Independent **negative** examples: CIFAR-10 test images

We sample 80 from each of the 10 test-set classes (800 images total). CIFAR-10 is 32×32 and often includes outdoor vegetation; a photograph with an animal in front of grass is not an unambiguous 'contains no plant' example. Interpret errors by visually examining the gallery; this is a robustness diagnostic, **not a camera-domain false acceptance estimate**.

In [ ]:
(_, _),(cifar_x,cifar_y)=tf.keras.datasets.cifar10.load_data()
cifar_y=cifar_y.reshape(-1)
cifar_names=['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
obj_rng=np.random.default_rng(SEED)
count_before=len(samples)
for cls,name in enumerate(cifar_names):
    indices=np.flatnonzero(cifar_y==cls)
    chosen_indices=np.sort(obj_rng.choice(indices,size=80,replace=False))
    for idx in chosen_indices:
        add_sample(cifar_x[idx],'CIFAR-10 / low-res objects',name,1,int(idx))
print('Non-plant foreground examples:',len(samples)-count_before)
assert len(samples)-count_before==800

## Cell 6 (optional) — Additional photos taken or collected by you

For a more realistic diagnostic, optionally upload one ZIP with images in folders `plant/` and `nonplant/`. **Do not label pictures that prominently contain both plant and non-plant subjects unless the expected behaviour is unambiguous.** Only manually labeled photos can meaningfully test the exact smartphone-camera conditions. If you have none, skip this cell; the rest runs without it.

Example ZIP structure:

```
field_photos.zip
├── plant/
│   ├── tomato_leaf.jpg
│   └── mango_leaf.jpg
└── nonplant/
    ├── bottle.jpg
    └── chair.jpg
```

**Privacy:** Avoid faces, identification documents, GPS metadata or private scenes. ZIPs are read into memory, not automatically published.

In [ ]:
# OPTIONAL: Set to True only if you want to upload manually labeled photos.
ADD_OWN_PHOTOS=False

if ADD_OWN_PHOTOS:
    uploaded=files.upload()
    assert len(uploaded)==1, 'Upload a single ZIP containing plant/ and nonplant/ folders'
    archive_name,raw=next(iter(uploaded.items()))
    assert archive_name.lower().endswith('.zip')
    with zipfile.ZipFile(io.BytesIO(raw)) as arch:
        count=0
        for member in arch.infolist():
            if member.is_dir() or member.file_size>20*1024**2:
                continue
            parts=Path(member.filename).parts
            labels=[p.lower() for p in parts if p.lower() in ('plant','nonplant')]
            if len(labels)!=1 or not member.filename.lower().endswith(('.jpg','.jpeg','.png','.webp')):
                continue
            label=labels[0]
            with Image.open(io.BytesIO(arch.read(member))) as im:
                im=ImageOps.exif_transpose(im).convert('RGB')
                add_sample(np.asarray(im), 'Manual field photos', label,
                           0 if label=='plant' else 1, count)
            count+=1
            assert count<=250, 'Limit your upload to 250 labeled images'
    print('Manual photos added:',count)
else:
    print('No manually labeled field photographs supplied')

## Cell 7 — Check external sample composition and duplicate hashes

We check **exact matching resized RGB bytes within the external stress set**. We cannot prove no overlap with the original Part 3 training images because those source images are not included in the Part 3 ZIP. The external datasets were not used in our Part 3 run.

In [ ]:
external=pd.DataFrame(samples)
assert len(external)==len(images)
assert set(external.Truth.unique())=={0,1}
# Retain a unique representative for exact duplicate preprocessed pixels.
keep=~external.duplicated('ImageSHA256_RGB160')
removed=int((~keep).sum())
idx_keep=np.flatnonzero(keep.to_numpy())
images=[images[i] for i in idx_keep]
external=external.loc[keep].reset_index(drop=True)
print('Exact duplicate RGB160 images removed:',removed)
print('Total external images:',len(external))
display(pd.crosstab(external.Source,external.Truth).rename(columns={0:'Plant',1:'Non-plant'}))
display(external.groupby(['Source','Category']).size().to_frame('Count'))

## Cell 8 — Run the frozen original Keras validator

Input range is **0–255 float32**. The model already performs MobileNetV2 normalization internally. Output is a **non-plant logit**, so `plant_score = sigmoid(-logit)`.

In [ ]:
all_logits=[]
for first in range(0,len(images),BATCH):
    batch=np.stack(images[first:first+BATCH]).astype(np.float32)
    batch_logits=model(batch,training=False).numpy().reshape(-1)
    all_logits.extend(batch_logits.tolist())
external['NonplantLogit']=np.asarray(all_logits,dtype=np.float32)
external['PlantScore']=expit(-external.NonplantLogit.to_numpy(dtype=np.float64))
assert np.isfinite(external.NonplantLogit.to_numpy()).all()
assert external.PlantScore.between(0,1).all()
print('Logit range:',external.NonplantLogit.min(),external.NonplantLogit.max())
print('Plant-score percentiles:')
display(external.groupby('Source')['PlantScore'].agg(['min','median','max']).round(6))

## Cell 9 — Compare the **two frozen thresholds** on independent images

*Plant retention* = proportion of labelled plant-like images that pass Stage A. *Object rejection* = proportion of labelled non-plant primary-object images that Stage A rejects. These are not supported-crop identification metrics. No threshold is selected here based on these data.

In [ ]:
rows=[]; per_source=[]
truth=external.Truth.to_numpy()
score=external.PlantScore.to_numpy()
for name,t in FROZEN_THRESHOLDS.items():
    accepts=score>=t
    p=(truth==0); n=(truth==1)
    rows.append({'Rule':name,'Threshold':t,'Plant images':int(p.sum()),
                 'Plant accepted':int((p&accepts).sum()),
                 'Plant retention (%)':100*np.mean(accepts[p]),
                 'Non-plant images':int(n.sum()),
                 'Non-plant rejected':int((n&~accepts).sum()),
                 'Non-plant rejection (%)':100*np.mean(~accepts[n]),
                 'Balanced (%)':50*(np.mean(accepts[p])+np.mean(~accepts[n]))*100})
    for source,part in external.groupby('Source'):
        idx=part.index.to_numpy(dtype=np.int32)
        desired=(truth[idx]==0)
        correct=(accepts[idx]==desired)
        per_source.append({'Rule':name,'Source':source,'Count':len(idx),
                           'Label':'Plant' if desired.all() else ('Non-plant' if (~desired).all() else 'Mixed'),
                           'Correct (%)':100*np.mean(correct),
                           'Pass as plant (%)':100*np.mean(accepts[idx])})
summary=pd.DataFrame(rows)
source_report=pd.DataFrame(per_source)
print('AUROC, nonplant as positive:',roc_auc_score(truth,external.NonplantLogit)*100)
display(summary.round(4))
display(source_report.round(4))
summary.to_csv(OUT/'external_threshold_summary.csv',index=False)
source_report.to_csv(OUT/'external_by_source.csv',index=False)
external.to_csv(OUT/'external_image_scores.csv',index=False)

## Cell 10 — Per-class errors and confidence distributions

In [ ]:
per_category=[]
for name,t in FROZEN_THRESHOLDS.items():
    accept=external.PlantScore.to_numpy()>=t
    for (source,cat),sub in external.groupby(['Source','Category']):
        idx=sub.index.to_numpy(dtype=np.int32)
        desired=(truth[idx]==0)
        per_category.append({'Rule':name,'Source':source,'Category':cat,'Count':len(idx),
                             'Correct (%)':100*np.mean(accept[idx]==desired),
                             'Accepted as plant (%)':100*np.mean(accept[idx])})
per_category=pd.DataFrame(per_category)
per_category.to_csv(OUT/'external_by_category.csv',index=False)
display(per_category.round(2))

fig,axes=plt.subplots(1,2,figsize=(14,4.5))
for ax,source in zip(axes,['Beans / independent field leaves','TF Flowers / external flowers']):
    s=external[external.Source==source]
    ax.hist(s.PlantScore,bins=np.linspace(0,1,41),color='seagreen',alpha=.8)
    for label,t in FROZEN_THRESHOLDS.items():
        ax.axvline(t,linestyle='--',label=label)
    ax.set(xlabel='Plant score',ylabel='Number of images',title=source)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT/'external_plant_score_distribution.png',dpi=160)
plt.show()

fig,ax=plt.subplots(figsize=(9,4))
for source,part in external.groupby('Source'):
    ax.hist(part.PlantScore,bins=np.linspace(0,1,51),alpha=.45,density=True,label=source)
for label,t in FROZEN_THRESHOLDS.items():
    ax.axvline(t,linestyle='--',label=label)
ax.set(xlabel='Plant score',ylabel='Density',title='External sources (diagnostic)')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT/'external_all_score_distributions.png',dpi=160)
plt.show()

## Cell 11 — Error gallery and inspection

Review visually; CIFAR-10 images may contain vegetation behind the named non-plant object. An 'error' under the foreground-object label may be defensible if the model sees substantial plants.

In [ ]:
t=float(config['accept_if_plant_score_at_least'])
accept=external.PlantScore.to_numpy()>=t
false_accept=np.flatnonzero((truth==1)&accept)
false_reject=np.flatnonzero((truth==0)&~accept)
# For false acceptance, show highest plant-confidence objects.
false_accept=false_accept[np.argsort(-score[false_accept])][:8]
# For false rejection, show lowest plant-confidence plants.
false_reject=false_reject[np.argsort(score[false_reject])][:8]
fig,axes=plt.subplots(2,8,figsize=(18,5.5))
for row,indices in enumerate([false_accept,false_reject]):
    for j,ax in enumerate(axes[row]):
        ax.axis('off')
        if j<len(indices):
            ix=int(indices[j]); r=external.iloc[ix]
            ax.imshow(images[ix])
            title=f'{r.Category}\nP(plant)={r.PlantScore:.3f}'
            ax.set_title(title,fontsize=8)
fig.suptitle('Top: false plant acceptance (objects); bottom: false plant rejection',fontsize=12)
fig.tight_layout()
fig.savefig(OUT/'external_failure_gallery.png',dpi=160)
plt.show()
print('False acceptance count (external negative):',int(((truth==1)&accept).sum()))
print('False rejection count (external positive):',int(((truth==0)&(~accept)).sum()))

## Cell 12 — Save results with provenance and download

**No fine-tuning, threshold adjustment or INT8 conversion is performed in this diagnostic notebook.** We should inspect source-specific performance first. Part 5 will use training-source images for INT8 calibration, not these evaluation images.

In [ ]:
audit={
 'stage':'Phase9_Part4_Independent_Stress_Test',
 'stageA_sha256':model_sha,
 'source_model_file':'crop_validator_stageA.keras',
 'thresholds_frozen_from_part3':FROZEN_THRESHOLDS,
 'total_external_images':int(len(external)),
 'external_by_source':external.Source.value_counts().to_dict(),
 'external_by_truth':{'Plant':int((truth==0).sum()),'NonplantForeground':int((truth==1).sum())},
 'evaluation_only':True,
 'thresholds_refit_on_external_data':False,
 'limitations':[
     'CIFAR-10 negatives are 32x32 and can include vegetation in background',
     'Flower photographs assess plant-like input, not suitability for disease diagnosis',
     'Some photos may contain ambiguous foreground/background subjects',
     'Exact duplicate screening against Part3 training photos was not possible from the model-only ZIP',
     'No real-world smartphone field photos unless explicitly uploaded in Cell 6'
 ]}
(OUT/'external_stress_audit.json').write_text(json.dumps(audit,indent=2))
# Do not duplicate the Keras model in results; user already has Part3 ZIP.
(OUT/'crop_validator_stageA.keras').unlink()
archive=shutil.make_archive(str(ROOT/'CropDetector_Phase9_Part4_Results'),'zip',str(OUT))
print('Result ZIP:',archive)
with zipfile.ZipFile(archive) as z:
    for m in z.infolist(): print(m.filename,round(m.file_size/1024,1),'KiB')
files.download(archive)

---
## What to send back

Upload **`CropDetector_Phase9_Part4_Results.zip`**. We will examine actual failures by source and determine whether the existing model is worth quantizing. If it underperforms on new leaf photographs, **do not change the threshold using the external evaluation set**; instead collect a *separate training/development set* for improving generalization and reserve new independent data for a fair retest.

If robustness is satisfactory for the prototype, Phase 9 Part 5 will create an INT8 validator using **representative training images from the saved Part 2 manifests**, evaluate parity, and integrate it ahead of the existing INT8 disease classifier. Stage B (supported crop recognition) remains a distinct task.